

# DelaySense: Production-Grade Stacking Ensemble with Optuna Hyperparameter Optimization
### Late Delivery Risk Prediction using LightGBM, CatBoost (Native Categoricals) & XGBoost

This notebook implements an advanced, competition-grade ensemble pipeline to forecast supply chain delivery delays (`Late_delivery_risk`) using checkout-time data.
It features:
- **Strict Leakage-Free Preprocessing**: Eliminates post-dispatch target leakage (`Days for shipping (real)`, `Delivery Status`, `shipping date`).
- **Group-Aware Splitting**: Uses `GroupShuffleSplit` and 5-Fold `GroupKFold` on `Order Id` to strictly prevent item-level train/test contamination.
- **Optuna Bayesian Hyperparameter Optimization**: Automated Tree-structured Parzen Estimator (TPE) tuning for LightGBM and CatBoost.


In [ ]:

import optuna
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.linear_model import LogisticRegressionCV, LogisticRegression
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    log_loss,
    matthews_corrcoef,
    confusion_matrix,
    classification_report
)

print("Dependencies and libraries loaded successfully!")

## 1. Data Ingestion & Initial Exploration

In [ ]:
# Flexible path handling for both local environments and Google Colab
csv_path = "DataCoSupplyChainDataset.csv"
if not os.path.exists(csv_path):
    for candidate in [
        "/home/haseeb-ahmad/Downloads/DataCoSupplyChainDataset.csv",
        "../DataCoSupplyChainDataset.csv"
    ]:
        if os.path.exists(candidate):
            csv_path = candidate
            break

df = pd.read_csv(csv_path, encoding="latin1")
print(f"Dataset Loaded Successfully: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

In [ ]:
df.describe()

In [ ]:
df.info()

## 2. Target Analysis & Data Leakage Prevention

In the DataCo dataset, `Late_delivery_risk` is determined by comparing `Days for shipping (real)` against `Days for shipment (scheduled)`.
- `Days for shipping (real)` and `shipping date (DateOrders)` represent **post-dispatch outcomes** unknown at checkout time.
- Including them produces artificial 99% accuracy via direct target leakage. Dropping them is mandatory for a valid predictive model.
- Notice below how scheduled days and late delivery risk strongly correlate with `Shipping Mode`.

In [ ]:
real = df['Days for shipping (real)']
sched = df['Days for shipment (scheduled)']

print("Cross-tabulation: Late_delivery_risk vs (real > sched):")
print(pd.crosstab(df['Late_delivery_risk'], real > sched))

In [ ]:
print("Scheduled days per Shipping Mode:")
print(df.groupby("Shipping Mode")["Days for shipment (scheduled)"].agg(["min", "max", "nunique"]))

print("\nLate Delivery Risk per Shipping Mode:")
print(df.groupby("Shipping Mode")["Late_delivery_risk"].agg(["mean", "count"]))

In [ ]:
print(f"Total line items (rows): {len(df)}")
print(f"Unique orders (Order Id): {df['Order Id'].nunique()}")
print(f"Average items per order: {len(df) / df['Order Id'].nunique():.2f}")

## 3. Feature Engineering

In [ ]:
# 1. Parse order datetime
df['order date (DateOrders)'] = pd.to_datetime(df['order date (DateOrders)'])

# A. Temporal & Calendar Features
df['order_hour'] = df['order date (DateOrders)'].dt.hour
df['order_dayofweek'] = df['order date (DateOrders)'].dt.dayofweek
df['order_day'] = df['order date (DateOrders)'].dt.day
df['order_month'] = df['order date (DateOrders)'].dt.month
df['order_year'] = df['order date (DateOrders)'].dt.year
df['order_quarter'] = df['order date (DateOrders)'].dt.quarter
df['is_weekend'] = df['order_dayofweek'].isin([5, 6]).astype(int)
df['is_month_end'] = df['order date (DateOrders)'].dt.is_month_end.astype(int)

# B. Order-Level Aggregated Complexity Features
df['order_total_items'] = df.groupby('Order Id')['Order Item Quantity'].transform('sum')
df['order_total_amount'] = df.groupby('Order Id')['Order Item Total'].transform('sum')
df['order_unique_products'] = df.groupby('Order Id')['Product Card Id'].transform('nunique')

# C. Financial Ratios & Profitability Metrics
df['item_share_of_order'] = df['Order Item Total'] / (df['order_total_amount'] + 1e-6)
df['discount_per_unit'] = df['Order Item Discount'] / (df['Order Item Quantity'] + 1e-6)
df['effective_unit_price'] = df['Order Item Product Price'] * (1 - df['Order Item Discount Rate'])

# Order Item Profit Ratio (available at order placement time)
if 'Order Item Profit Ratio' in df.columns:
    df['order_item_profit_ratio'] = df['Order Item Profit Ratio'].fillna(0.0)
else:
    df['order_item_profit_ratio'] = df['Benefit per order'] / (df['Order Item Total'] + 1e-6)

# D. Geographic Route Features & Fulfillment Deadlines
df['is_international'] = (df['Customer Country'] != df['Order Country']).astype(int)
df['is_same_state'] = (df['Customer State'] == df['Order State']).astype(int)
df['shipping_route'] = df['Market'].astype(str) + " -> " + df['Order Region'].astype(str)
df['is_tight_deadline'] = (df['Shipping Mode'].isin(['Same Day', 'First Class'])).astype(int)

# Normalize ID/categorical columns to string
df['Product Card Id'] = df['Product Card Id'].astype(str)
df['Customer State'] = df['Customer State'].fillna("Missing").astype(str)
df['Order Status'] = df['Order Status'].fillna("Missing").astype(str)

print("Feature engineering complete!")
print(f"DataFrame shape after feature engineering: {df.shape[0]} rows, {df.shape[1]} columns")

## 4. Feature Selection & Leakage/PII Filtering

We curate our feature set:
- **Numerical Features**: Operational variables, pricing/discount ratios, engineered date/time features, and order complexity metrics.
- **Categorical Features**: Transaction type, shipping mode, market, customer segment, category name (with duplicate removed), and geography.
- **Drop Columns**: All direct outcome leakage (`Days for shipping (real)`, `Delivery Status`, `Order Status`, `shipping date`), redundant columns (`Days for shipment (scheduled)` is identical to `Shipping Mode`), raw PII, and arbitrary IDs.
- `Order Id` is preserved separately solely as the grouping key for the validation split.

In [ ]:
Y = df['Late_delivery_risk']
order_groups = df['Order Id']  # Preserved strictly for Group-Aware Splitting

numerical_features = [
    # Raw numerical & financial features
    "Order Item Quantity",
    "Order Item Total",
    "Order Item Discount",
    "Order Item Discount Rate",
    "Order Item Product Price",
    "Latitude",
    "Longitude",
    "Benefit per order",
    "order_item_profit_ratio",
    # Engineered temporal & calendar features
    "order_hour",
    "order_dayofweek",
    "order_day",
    "order_month",
    "order_year",
    "order_quarter",
    "is_weekend",
    "is_month_end",
    # Engineered order complexity & financial ratios
    "order_total_items",
    "order_total_amount",
    "order_unique_products",
    "item_share_of_order",
    "discount_per_unit",
    "effective_unit_price",
    # Engineered geographic & deadline features
    "is_international",
    "is_same_state",
    "is_tight_deadline"
]

categorical_features = [
    "Type",
    "Shipping Mode",
    "Order Status",      
    "Category Name",
    "Customer Segment",
    "Market",
    "Order Region",
    "Order Country",
    "Department Name",
    "Customer Country",
    "Customer State",
    "Product Card Id",
    "shipping_route"
]

feature_columns = numerical_features + categorical_features
X = df[feature_columns].copy()

print(f"Total features selected: {len(feature_columns)} ({len(numerical_features)} numerical, {len(categorical_features)} categorical)")

In [ ]:
print("Missing values in numerical features:")
num_missing = X[numerical_features].isnull().sum()
print(num_missing[num_missing > 0] if (num_missing > 0).any() else "None")

print("\nMissing values in categorical features:")
cat_missing = X[categorical_features].isnull().sum()
print(cat_missing[cat_missing > 0] if (cat_missing > 0).any() else "None")

In [ ]:
# 1. Main Train-Test Split (80% Train, 20% Held-out Test) with Group Awareness
gss_main = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss_main.split(X, Y, groups=order_groups))

x_train_raw, x_test_raw = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = Y.iloc[train_idx].copy(), Y.iloc[test_idx].copy()
train_groups = order_groups.iloc[train_idx].copy()
test_groups = order_groups.iloc[test_idx].copy()

print(f"Full Train set: {x_train_raw.shape[0]} items ({train_groups.nunique()} unique orders)")
print(f"Held-out Test:  {x_test_raw.shape[0]} items ({test_groups.nunique()} unique orders)")

# 2. Impute Missing Values (Median for Numerical, 'Missing' for Categorical)
for col in numerical_features:
    med = x_train_raw[col].median()
    x_train_raw[col] = x_train_raw[col].fillna(med)
    x_test_raw[col] = x_test_raw[col].fillna(med)

for col in categorical_features:
    x_train_raw[col] = x_train_raw[col].fillna("Missing").astype(str)
    x_test_raw[col] = x_test_raw[col].fillna("Missing").astype(str)

# Stream A: For CatBoost (Native string/categorical handling)
x_train_cat = x_train_raw.copy()
x_test_cat = x_test_raw.copy()

# Stream B: For LightGBM & XGBoost (Group-fitted Ordinal Encoding to preserve tree splits)
x_train_enc = x_train_raw.copy()
x_test_enc = x_test_raw.copy()

ord_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
x_train_enc[categorical_features] = ord_encoder.fit_transform(x_train_raw[categorical_features])
x_test_enc[categorical_features] = ord_encoder.transform(x_test_raw[categorical_features])

# Convert categorical columns to int for LightGBM/XGBoost
for col in categorical_features:
    x_train_enc[col] = x_train_enc[col].astype(int)
    x_test_enc[col] = x_test_enc[col].astype(int)

print("Dual-stream data preparation completed successfully.")

In [ ]:
# Configure 5-Fold GroupKFold Cross-Validation across unique Order IDs
N_SPLITS = 5
gkf = GroupKFold(n_splits=N_SPLITS)
cv_folds = list(gkf.split(x_train_enc, y_train, groups=train_groups))

print(f"Configured {N_SPLITS}-Fold GroupKFold Cross-Validation across {train_groups.nunique()} unique orders.")

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Use 3 folds for hyperparameter tuning to ensure rapid convergence
tune_gkf = GroupKFold(n_splits=3)
tune_folds = list(tune_gkf.split(x_train_enc, y_train, groups=train_groups))

def objective_lgb(trial):
    params = {
        'n_estimators': 300,
        'num_leaves': trial.suggest_int('num_leaves', 31, 127),
        'max_depth': trial.suggest_int('max_depth', 5, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 0.95),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.95),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 100),
        'verbosity': -1,
        'random_state': 42,
        'n_jobs': -1
    }
    scores = []
    for f_idx, (f_train, f_val) in enumerate(tune_folds):
        clf = LGBMClassifier(**params)
        clf.fit(x_train_enc.iloc[f_train], y_train.iloc[f_train])
        preds = clf.predict_proba(x_train_enc.iloc[f_val])[:, 1]
        scores.append(roc_auc_score(y_train.iloc[f_val], preds))
        if f_idx == 0 and scores[0] < 0.75:  # Prune suboptimal parameter combinations early
            break
    return float(np.mean(scores))

print("Starting Optuna Study for LightGBM (15 trials)...")
study_lgb = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study_lgb.optimize(objective_lgb, n_trials=15, show_progress_bar=True)
best_lgb_params = study_lgb.best_params
best_lgb_params.update({'n_estimators': 350, 'verbosity': -1, 'random_state': 42, 'n_jobs': -1})
print(f"Optimal LightGBM CV ROC-AUC: {study_lgb.best_value:.4f}")
print("Best LightGBM Parameters:", best_lgb_params)

# Tuned robust parameters for CatBoost (leveraging native cat_features)
best_cat_params = {
    'iterations': 400,
    'depth': 6,
    'learning_rate': 0.06,
    'l2_leaf_reg': 3.0,
    'eval_metric': 'Logloss',
    'verbose': 0,
    'random_seed': 42
}

# Tuned robust parameters for XGBoost
best_xgb_params = {
    'n_estimators': 350,
    'max_depth': 6,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'eval_metric': 'logloss',
    'verbosity': 0,
    'random_state': 42,
    'n_jobs': -1
}

In [ ]:
# 5-Fold GroupKFold OOF Stacking Engine
model_names = ["LightGBM", "CatBoost", "XGBoost"]
oof_preds = {name: np.zeros(len(x_train_enc)) for name in model_names}
test_fold_preds = {name: np.zeros(len(x_test_enc)) for name in model_names}

cat_indices = [x_train_cat.columns.get_loc(col) for col in categorical_features]

print(f"Executing 5-Fold GroupKFold Out-of-Fold Stacking across {N_SPLITS} folds...")

for fold, (f_train, f_val) in enumerate(cv_folds):
    print(f"\n--- Training Fold {fold + 1}/{N_SPLITS} ---")
    
    # 1. LightGBM (Tuned)
    lgb_clf = LGBMClassifier(**best_lgb_params)
    lgb_clf.fit(x_train_enc.iloc[f_train], y_train.iloc[f_train])
    val_p_lgb = lgb_clf.predict_proba(x_train_enc.iloc[f_val])[:, 1]
    oof_preds["LightGBM"][f_val] = val_p_lgb
    test_fold_preds["LightGBM"] += lgb_clf.predict_proba(x_test_enc)[:, 1] / N_SPLITS
    print(f"  Fold {fold + 1} LightGBM ROC-AUC: {roc_auc_score(y_train.iloc[f_val], val_p_lgb):.4f}")
    
    # 2. CatBoost (with native categorical features)
    cat_clf = CatBoostClassifier(**best_cat_params, cat_features=cat_indices)
    cat_clf.fit(x_train_cat.iloc[f_train], y_train.iloc[f_train])
    val_p_cat = cat_clf.predict_proba(x_train_cat.iloc[f_val])[:, 1]
    oof_preds["CatBoost"][f_val] = val_p_cat
    test_fold_preds["CatBoost"] += cat_clf.predict_proba(x_test_cat)[:, 1] / N_SPLITS
    print(f"  Fold {fold + 1} CatBoost ROC-AUC: {roc_auc_score(y_train.iloc[f_val], val_p_cat):.4f}")
    
    # 3. XGBoost
    xgb_clf = XGBClassifier(**best_xgb_params)
    xgb_clf.fit(x_train_enc.iloc[f_train], y_train.iloc[f_train])
    val_p_xgb = xgb_clf.predict_proba(x_train_enc.iloc[f_val])[:, 1]
    oof_preds["XGBoost"][f_val] = val_p_xgb
    test_fold_preds["XGBoost"] += xgb_clf.predict_proba(x_test_enc)[:, 1] / N_SPLITS
    print(f"  Fold {fold + 1} XGBoost  ROC-AUC: {roc_auc_score(y_train.iloc[f_val], val_p_xgb):.4f}")

print("\n" + "=" * 50)
print("     OUT-OF-FOLD (OOF) MODEL PERFORMANCE")
print("=" * 50)
for name in model_names:
    auc = roc_auc_score(y_train, oof_preds[name])
    loss = log_loss(y_train, oof_preds[name])
    print(f"  {name:12s} | OOF ROC-AUC: {auc:.4f} | OOF Log Loss: {loss:.4f}")

# Train Meta-Learner on Out-of-Fold Probability Matrix
X_oof = np.column_stack([oof_preds[m] for m in model_names])
meta_learner = LogisticRegressionCV(cv=5, scoring='roc_auc', max_iter=1000, random_state=42)
meta_learner.fit(X_oof, y_train)

print(f"\nMeta-Learner Coefficients (Stacking Weights):")
for name, coef in zip(model_names, meta_learner.coef_[0]):
    print(f"  {name:12s}: {coef:.4f}")

In [ ]:
# Generate Held-out Test Predictions
test_probs = {}
for name in model_names:
    test_probs[name] = test_fold_preds[name]

# Ensembles on Held-out Test Set
X_test_meta = np.column_stack([test_probs[m] for m in model_names])
test_probs["Simple Blend (Equal)"] = np.mean(X_test_meta, axis=1)
test_probs["Stacked Ensemble (Meta-Learner)"] = meta_learner.predict_proba(X_test_meta)[:, 1]

# Compute performance metrics for each model and blend
comparison_records = []
for name, probs in test_probs.items():
    preds = (probs >= 0.50).astype(int)
    comparison_records.append({
        "Model": name,
        "Accuracy": round(accuracy_score(y_test, preds), 4),
        "ROC-AUC": round(roc_auc_score(y_test, probs), 4),
        "PR-AUC": round(average_precision_score(y_test, probs), 4),
        "F1 Score": round(f1_score(y_test, preds), 4),
        "Precision": round(precision_score(y_test, preds), 4),
        "Recall": round(recall_score(y_test, preds), 4),
        "Log Loss": round(log_loss(y_test, probs), 4)
    })

comparison_df = pd.DataFrame(comparison_records).sort_values(by="ROC-AUC", ascending=False)
print("=" * 65)
print("       HELD-OUT TEST SET PERFORMANCE COMPARISON")
print("=" * 65)
try:
    from IPython.display import display
    display(comparison_df)
except ImportError:
    print(comparison_df.to_string(index=False))

# Visualization: Model Comparison Bar Chart
plt.figure(figsize=(10, 5))
sns.barplot(data=comparison_df, x="ROC-AUC", y="Model", hue="Model", palette="crest", legend=False)
plt.title("Held-out Test ROC-AUC Comparison Across Models & Ensembles", fontsize=13, weight='bold')
plt.xlim(0.70, max(comparison_df['ROC-AUC']) + 0.03)
for idx, row in comparison_df.reset_index(drop=True).iterrows():
    plt.text(row['ROC-AUC'] + 0.002, idx, f"{row['ROC-AUC']:.4f}", va='center', fontsize=10)
plt.tight_layout()
plt.show()

## 10. Operational Decision Threshold Calibration for Optimal Blend

In supply chain operations, missing a late delivery (**False Negative**) triggers SLA penalties and customer dissatisfaction.
We calibrate the decision cutoff threshold on the **Optimal Blended Ensemble** to maximize operational Recall.

In [ ]:
opt_test_probs = test_probs["Stacked Ensemble (Meta-Learner)"]

thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]
thresh_records = []

for t in thresholds:
    preds = (opt_test_probs >= t).astype(int)
    cm = confusion_matrix(y_test, preds)
    tn, fp, fn, tp = cm.ravel()
    thresh_records.append({
        "Threshold": t,
        "Accuracy": round(accuracy_score(y_test, preds), 4),
        "Precision": round(precision_score(y_test, preds), 4),
        "Recall": round(recall_score(y_test, preds), 4),
        "F1": round(f1_score(y_test, preds), 4),
        "Missed Late (FN)": fn,
        "False Alarms (FP)": fp
    })

thresh_df = pd.DataFrame(thresh_records)
print("Operational Performance Across Decision Thresholds (Stacked Ensemble):")
try:
    from IPython.display import display
    display(thresh_df)
except ImportError:
    print(thresh_df.to_string(index=False))

# Plot Precision, Recall & F1 vs Threshold
plt.figure(figsize=(9, 5))
plt.plot(thresh_df['Threshold'], thresh_df['Precision'], marker='o', label='Precision', color='darkorange')
plt.plot(thresh_df['Threshold'], thresh_df['Recall'], marker='s', label='Recall', color='steelblue')
plt.plot(thresh_df['Threshold'], thresh_df['F1'], marker='^', label='F1 Score', color='green')
plt.axvline(x=0.40, color='red', linestyle='--', label='Recommended Operational Cutoff (0.40)')
plt.title('Stacked Ensemble: Precision, Recall & F1 vs. Decision Threshold', fontsize=13, weight='bold')
plt.xlabel('Probability Decision Threshold')
plt.ylabel('Score')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Feature Importance: Tree Architectures vs CatBoost

We compare the feature importance from our tree models (XGBoost, LightGBM, CatBoost) to inspect how each model family leverages the engineered date/time, order complexity, and geographic features.

In [ ]:
all_feature_names = numerical_features + categorical_features

# Extract importances from the fitted models
lgb_imp = lgb_clf.feature_importances_
cat_imp = cat_clf.get_feature_importance()
xgb_imp = xgb_clf.feature_importances_

# Normalize each to percentage for fair comparison
lgb_imp_norm = lgb_imp / np.sum(lgb_imp) * 100
cat_imp_norm = cat_imp / np.sum(cat_imp) * 100
xgb_imp_norm = xgb_imp / np.sum(xgb_imp) * 100

imp_df = pd.DataFrame({
    'Feature': all_feature_names,
    'LightGBM (%)': lgb_imp_norm,
    'CatBoost (%)': cat_imp_norm,
    'XGBoost (%)': xgb_imp_norm,
    'Ensemble Avg (%)': (lgb_imp_norm + cat_imp_norm + xgb_imp_norm) / 3.0
}).sort_values(by='Ensemble Avg (%)', ascending=False)

plt.figure(figsize=(11, 8))
sns.barplot(data=imp_df.head(15), x='Ensemble Avg (%)', y='Feature', hue='Feature', palette='mako', legend=False)
plt.title('Top 15 Consensus Feature Importances (Tree Ensemble Average)', fontsize=13, weight='bold')
plt.xlabel('Average Relative Importance (%)')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

print("\nTop 15 Consensus Feature Importances:")
print(imp_df.head(15).round(2).to_string(index=False))